## Декораторы


In [2]:
def greet(name):
    return f"Привет, {name}!"

# Присваиваем функцию переменной
my_function = greet
print(greet)
print(my_function)
print(my_function("Bob"))

<function greet at 0x110c058a0>
<function greet at 0x110c058a0>
Привет, Bob!


In [3]:
def apply_operation(func, x, y):
    return func(x, y)

def add(a, b):
    return a + b

def multiply(a, b):
    return a * b

print(apply_operation(add, 5, 3))
print(apply_operation(multiply, 5, 3))

8
15


In [4]:
def create_multiplier(n):
    def multiplier(x):
        return x * n
    return multiplier

double = create_multiplier(2)
triple = create_multiplier(3)

print(double(5))
print(triple(5))

10
15


In [5]:
def decorator(func):  # 1. Декоратор принимает функцию
    def wrapper():    # 2. Внутренняя функция-обёртка
        print("До вызова функции")
        func()        # 3. Вызов оригинальной функции
        print("После вызова функции")
    return wrapper    # 4. Возвращаем обёртку


def hello():
    print("Привет!")



hello()

Привет!


In [6]:
def decorator(func):
    def wrapper(*args, **kwargs):
        print("Что-то делаем до")
        result = func(*args, **kwargs)  # Передаём аргументы
        print("Что-то делаем после")
        return result
    return wrapper

@decorator
def sum(a, b):
    return a + b

print(sum(2, 3))

Что-то делаем до
Что-то делаем после
5


In [ ]:
@decorator1
@decorator2
def my_func():
    pass

In [ ]:
my_func = decorator1(decorator2(my_func))

In [7]:
def tag(name):
    def deco(func):
        def wrapper(*args, **kwargs):
            print(f"<{name}>")
            result = func(*args, **kwargs)
            print(f"</{name}>")
            return result
        return wrapper
    return deco

@tag("b")
@tag("i")
def text():
    print("привет")

text()

<b>
<i>
привет
</i>
</b>


In [8]:
import time

def timer(func):
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"Время выполнения: {time.time() - start:.2f} сек.")
        return result
    return wrapper

@timer
def slow_func():
    time.sleep(1)

slow_func()

Время выполнения: 1.01 сек.


In [9]:
def logger(func):
    def wrapper(*args, **kwargs):
        print(f"Вызов функции {func.__name__} с аргументами {args}")
        return func(*args, **kwargs)
    return wrapper

@logger
def multiply(a, b):
    return a * b

multiply(3, 4)

Вызов функции multiply с аргументами (3, 4)


12

## Контекстные менеджеры

In [1]:
# Простой контекстный менеджер через класс
import time

class Timer:
    def __enter__(self):
        self.start = time.time()
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        self.elapsed = time.time() - self.start
        print(f"Выполнено за {self.elapsed:.4f} сек")
        return False

with Timer():
    total = sum(range(1_000_000))
    print(f"Сумма: {total}")


Сумма: 499999500000
Выполнено за 0.0148 сек


In [2]:
with Timer():
    print(1 / 0)

Выполнено за 0.0000 сек


ZeroDivisionError: division by zero

In [3]:
from contextlib import suppress

with suppress(FileNotFoundError):
    open("нет_такого.txt")
print("живём дальше")

with suppress(ZeroDivisionError):
    print("до")
    print(1 / 0)
    print("после")  # не выполнится
print("конец")

живём дальше
до
конец


In [4]:
# Контекстный менеджер через декоратор
from contextlib import contextmanager
import os
import tempfile
import shutil

@contextmanager
def atomic_write(filename):
    """Безопасная запись - если ошибка, оригинальный файл не изменится"""
    temp_fd, temp_path = tempfile.mkstemp(
        dir=os.path.dirname(filename) or '.',
        prefix='.tmp_'
    )
    
    try:
        with os.fdopen(temp_fd, 'w') as f:
            yield f
        shutil.move(temp_path, filename)
        print(f"Файл {filename} успешно записан")
    except Exception as e:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        print(f"Ошибка: {e}")
        raise

# Успешная запись
with atomic_write('data.txt') as f:
    f.write('Важные данные\n')


Файл data.txt успешно записан


In [5]:
# Запись с ошибкой - файл не изменится
try:
    with atomic_write('data.txt') as f:
        f.write('Начало...\n')
        raise ValueError("Ошибка!")
        f.write('Эта строка не запишется\n')
except ValueError:
    print("Исключение поймано, оригинальный файл не поврежден")


Ошибка: Ошибка!
Исключение поймано, оригинальный файл не поврежден


## Генераторы и эффективность памяти


In [6]:
# Разница в памяти: список vs генератор
import sys

list_numbers = [x for x in range(1_000_000)]
print(f"Список (1М элементов): {sys.getsizeof(list_numbers):,} байт")
print(f"Это примерно: {sys.getsizeof(list_numbers) / 1024 / 1024:.2f} МБ")

gen_numbers = (x for x in range(1_000_000))
print(f"Генератор: {sys.getsizeof(gen_numbers):,} байт")
print(f"Разница в {sys.getsizeof(list_numbers) / sys.getsizeof(gen_numbers):.0f} раз")

del list_numbers, gen_numbers


Список (1М элементов): 8,448,728 байт
Это примерно: 8.06 МБ
Генератор: 192 байт
Разница в 44004 раз


In [7]:
# Создаем большой тестовый файл
print("Создаем файл (1М строк)...")
with open('large_file.txt', 'w') as f:
    for i in range(1_000_000):
        f.write(f"Line {i}: Some data\n")

file_size = os.path.getsize('large_file.txt')
print(f"Размер: {file_size / 1024 / 1024:.2f} МБ")


Создаем файл (1М строк)...
Размер: 21.83 МБ


In [8]:
def count_lines_bad(filename):
    with open(filename) as f:
        lines = f.readlines()
    return len(lines)

def count_lines_good(filename):
    with open(filename) as f:
        return sum(1 for _ in f)

count = count_lines_good('large_file.txt')
print(f"Строк: {count:,}")


Строк: 1,000,000


In [9]:
# Генератор для фильтрации
def filter_lines(filename, pattern):
    """Возвращает только строки с pattern"""
    with open(filename) as f:
        for line in f:
            if pattern in line:
                yield line.strip()

# Создаем тестовый лог
with open('app.log', 'w') as f:
    for i in range(100):
        if i % 10 == 0:
            f.write(f"[ERROR] Error at step {i}\n")
        else:
            f.write(f"[INFO] Processing {i}\n")

# Используем генератор
print("Первые 3 ERROR:")
for i, error in enumerate(filter_lines('app.log', 'ERROR')):
    if i < 3:
        print(f"{error}")
    else:
        break


Первые 3 ERROR:
[ERROR] Error at step 0
[ERROR] Error at step 10
[ERROR] Error at step 20


In [10]:
# Батч-обработка
from itertools import islice

def batch_generator(filename, batch_size=1000):
    """Возвращает файл батчами"""
    with open(filename) as f:
        while True:
            batch = list(islice(f, batch_size))
            if not batch:
                break
            yield batch

# Обрабатываем батчами
batch_count = 0
total_lines = 0

for batch in batch_generator('large_file.txt', batch_size=1000):
    batch_count += 1
    total_lines += len(batch)
    if batch_count <= 3:
        print(f"Батч {batch_count}: {len(batch)} строк")

print(f"Всего: {batch_count} батчей, {total_lines:,} строк")


Батч 1: 1000 строк
Батч 2: 1000 строк
Батч 3: 1000 строк
Всего: 1000 батчей, 1,000,000 строк


In [11]:
# Цепочка генераторов (pipeline)
def read_lines(filename):
    with open(filename) as f:
        for line in f:
            yield line

def filter_by_pattern(lines, pattern):
    for line in lines:
        if pattern in line:
            yield line

def transform_upper(lines):
    for line in lines:
        yield line.upper()

def take_first(lines, n):
    for i, line in enumerate(lines):
        if i >= n:
            break
        yield line

pipeline = take_first(
    transform_upper(
        filter_by_pattern(
            read_lines('app.log'),
            'ERROR'
        )
    ),
    3
)

print("Результат pipeline:")
for line in pipeline:
    print(f"{line.strip()}")


Результат pipeline:
[ERROR] ERROR AT STEP 0
[ERROR] ERROR AT STEP 10
[ERROR] ERROR AT STEP 20
